In [ ]:
## LINK SHARED GOOGLE DRIVE
## Useful for the metadata nad .pth files, as well as saving downloads
from google.colab import drive
drive.mount('/content/drive')

import os

# Data Paths
SHARED_ROOT = "/content/drive/Shared drives/NPB 136 -- EYEBROW PIERCINGS"
STORAGE_FOLDER  = os.path.join(SHARED_ROOT, "Project File Storage")
TENSOR_FOLDER = os.path.join(STORAGE_FOLDER, "Tensors")
os.makedirs(TENSOR_FOLDER, exist_ok=True)

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.decomposition import PCA

# ── Load metadata ──────────────────────────────────────────────────────────────
meta = json.load(open(os.path.join(TENSOR_FOLDER, "metadata.json")))
df   = pd.DataFrame(meta)
df["patient"] = df["patient_code"].apply(lambda x: x.split("/")[2].rsplit("_", 1)[0])

# ── Load all tensors → flatten to feature matrix ───────────────────────────────
print("Loading tensors...")
X_raw, y_raw, patients = [], [], []

for entry in meta:
    t = torch.load(os.path.join(TENSOR_FOLDER, entry["file"]),
                   weights_only=True)          # (2, 600, 600)
    X_raw.append(t.numpy().flatten())          # 720,000 features
    y_raw.append(entry["label"])
    patients.append(df.loc[df["file"]==entry["file"], "patient"].values[0])

X_raw     = np.array(X_raw,     dtype=np.float32)  # (727, 720000)
y_raw     = np.array(y_raw,     dtype=np.int32)     # (727,)
patients  = np.array(patients)
print(f"X shape: {X_raw.shape}")

# ── PCA reduction (720k → 256 dims) ───────────────────────────────────────────
# Raw pixels are too high-dimensional for SVM/LR — reduce first
print("Fitting PCA...")
pca   = PCA(n_components=256, random_state=42)
X_pca = pca.fit_transform(X_raw)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

scaler = StandardScaler()
X      = scaler.fit_transform(X_pca)   # (727, 256) — scaled PCA features

# ── Summary statistics features (lightweight alternative) ─────────────────────
def extract_stats(tensor_flat, n_channels=2, h=600, w=600):
    """Per-channel: mean, std, p5, p25, p50, p75, p95 → 14 features"""
    img = tensor_flat.reshape(n_channels, h, w)
    feats = []
    for c in range(n_channels):
        ch = img[c].flatten()
        feats.extend([
            ch.mean(), ch.std(),
            np.percentile(ch, 5),  np.percentile(ch, 25),
            np.percentile(ch, 50), np.percentile(ch, 75),
            np.percentile(ch, 95),
        ])
    return feats

X_stats = np.array([extract_stats(x) for x in X_raw], dtype=np.float32)  # (727, 14)
X_stats_scaled = StandardScaler().fit_transform(X_stats)

print(f"Stats feature shape: {X_stats.shape}")

Loading tensors...
X shape: (727, 720000)
Fitting PCA...
PCA explained variance: 0.634
Stats feature shape: (727, 14)


In [ ]:
# ── Classical ML models ────────────────────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.1),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)":           SVC(kernel="rbf", C=1.0, probability=True),
    "XGBoost":             GradientBoostingClassifier(n_estimators=200, random_state=42),
}

# ── PART 1: Naive 5-fold CV (data leakage — for comparison only) ───────────────
print("\n" + "="*60)
print("PART 1: Naive 5-Fold CV (patient leakage — baseline only)")
print("="*60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    scores = cross_validate(clf, X, y_raw, cv=cv,
                            scoring=["accuracy", "f1", "roc_auc"],
                            return_train_score=True)
    print(f"\n{name}")
    print(f"  Train Acc: {scores['train_accuracy'].mean():.3f} ± {scores['train_accuracy'].std():.3f}")
    print(f"  Val Acc:   {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
    print(f"  Val AUC:   {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")


PART 1: Naive 5-Fold CV (patient leakage — baseline only)

Logistic Regression
  Train Acc: 0.894 ± 0.020
  Val Acc:   0.604 ± 0.030
  Val AUC:   0.619 ± 0.035

Random Forest
  Train Acc: 1.000 ± 0.000
  Val Acc:   0.601 ± 0.057
  Val AUC:   0.662 ± 0.056

SVM (RBF)
  Train Acc: 0.984 ± 0.005
  Val Acc:   0.611 ± 0.041
  Val AUC:   0.652 ± 0.051

XGBoost
  Train Acc: 1.000 ± 0.000
  Val Acc:   0.596 ± 0.014
  Val AUC:   0.645 ± 0.034


In [ ]:
# ── PART 2: Paired Patient Holdout CV ─────────────────────────────────────────
print("\n" + "="*60)
print("PART 2: Paired Patient Holdout CV (correct evaluation)")
print("="*60)

pd_patients = sorted(df[df["condition"]=="PD"]["patient"].unique(),
                     key=lambda x: int(x[2:]))
hc_patients = sorted(df[df["condition"]=="HC"]["patient"].unique(),
                     key=lambda x: int(x[2:]))

n_pairs = min(len(pd_patients), len(hc_patients))
folds   = list(zip(pd_patients[:n_pairs], hc_patients))
print(f"Using {n_pairs} paired folds (PD14 stays in train only)")

results = {name: [] for name in models}

for fold_idx, (pd_p, hc_p) in enumerate(folds):
    test_mask  = np.isin(patients, [pd_p, hc_p])
    train_mask = ~test_mask

    X_train, y_train = X[train_mask], y_raw[train_mask]
    X_test,  y_test  = X[test_mask],  y_raw[test_mask]

    for name, clf in models.items():
        clf.fit(X_train, y_train)
        y_prob = clf.predict_proba(X_test)[:, 1]
        auc    = roc_auc_score(y_test, y_prob)
        acc    = accuracy_score(y_test, clf.predict(X_test))
        results[name].append(auc)
        print(f"  F{fold_idx+1} {pd_p}+{hc_p} | {name[:12]:12s} AUC={auc:.3f} Acc={acc:.3f}")

print("\nPaired Holdout Summary:")
print("-"*50)
for name, aucs in results.items():
    print(f"{name:25s}  AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

In [ ]:
# ── PART 3: CNN ────────────────────────────────────────────────────────────────
class CellCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=7, stride=2, padding=3),  # (16,300,300)
            nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(3, stride=2),                              # (16,149,149)

            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2), # (32,75,75)
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(3, stride=2),                              # (32,37,37)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),            # (64,37,37)
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),                           # (64,4,4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*4*4, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class TensorDS(Dataset):
    def __init__(self, meta, folder, indices):
        self.entries = [meta[i] for i in indices]
        self.folder  = folder

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        x = torch.load(os.path.join(self.folder, e["file"]),
                       weights_only=True).float()           # (2,600,600)
        # Per-image normalization
        mean = x.mean(dim=[1,2], keepdim=True)
        std  = x.std(dim=[1,2],  keepdim=True).clamp(min=1e-8)
        x    = (x - mean) / std
        x    = F.interpolate(x.unsqueeze(0), size=(224,224),
                             mode="bilinear", align_corners=False).squeeze(0)
        return x, torch.tensor(e["label"], dtype=torch.float32)


def train_cnn_fold(train_idx, test_idx, meta, folder, device, epochs=15):
    train_ds = TensorDS(meta, folder, train_idx)
    test_ds  = TensorDS(meta, folder, test_idx)
    train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
    test_dl  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

    model     = CellCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()

    # Evaluate
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in test_dl:
            probs = torch.sigmoid(model(x.to(device))).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y.numpy())

    return roc_auc_score(all_labels, all_probs), \
           accuracy_score(all_labels, np.array(all_probs) > 0.5)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n{'='*60}")
print(f"PART 3: CNN — Paired Patient Holdout CV  (device={device})")
print(f"{'='*60}")

indices = np.arange(len(meta))
cnn_aucs = []

for fold_idx, (pd_p, hc_p) in enumerate(folds):
    test_mask  = np.isin(patients, [pd_p, hc_p])
    train_idx  = indices[~test_mask].tolist()
    test_idx   = indices[test_mask].tolist()

    auc, acc = train_cnn_fold(train_idx, test_idx, meta, TENSOR_FOLDER, device)
    cnn_aucs.append(auc)
    print(f"  F{fold_idx+1} {pd_p}+{hc_p}  AUC={auc:.3f}  Acc={acc:.3f}")

print(f"\nCNN Paired Holdout:  AUC: {np.mean(cnn_aucs):.3f} ± {np.std(cnn_aucs):.3f}")


PART 3: CNN — Paired Patient Holdout CV  (device=cuda)
  F1 PD1+HC1  AUC=0.062  Acc=0.259
  F2 PD2+HC2  AUC=0.487  Acc=0.500
  F3 PD3+HC3  AUC=0.645  Acc=0.491
  F4 PD4+HC4  AUC=0.111  Acc=0.389
  F5 PD5+HC5  AUC=0.332  Acc=0.415
  F6 PD6+HC6  AUC=0.816  Acc=0.741
  F7 PD7+HC7  AUC=0.997  Acc=0.519
